# Data Preprocessing 6 - The Complete Workflow

> **MLCourse · Data Science Foundations · 05_data_preprocessing**

Everything from this section in one continuous, realistic job: a deliberately
MESSY telecom-churn table gets walked through the professional 10-step checklist
until it is model-ready. Reusable utility functions included.

## What you'll learn
- A repeatable end-to-end order of operations (and WHY that order)
- Fixing mixed date formats, currency strings, casing chaos
- Deduplication by business key
- Per-column missing strategies with justification
- Outlier capping with IQR fences
- Encoding + scaling - done leak-free with train/test split
- Lightweight text features from a feedback column
- Saving the processed artifact

In [1]:
%matplotlib inline
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt

np.random.seed(42)
pd.options.display.width = 130

## 0. Build the mess - synthetic raw data

Every pathology below appears in real exports: mixed formats, whitespace,
impossible values, duplicates, free-text noise.

In [2]:
n = 300

def messy_money(x):
    """Render a float as one of several ugly currency strings."""
    if np.random.rand() < 0.15:
        return f"${x:,.2f}"          # $1,299.00 style
    return f"{x:.2f}"

ids = [f"cust-{i:03d}" for i in range(n)]
some_ids = np.random.choice(ids, size=int(n*0.04), replace=False)  # duplicate ids

raw = pd.DataFrame({
    "customer_id": list(ids) + list(some_ids),
    "signup_date": [
        np.random.choice([
            pd.Timestamp("2022-01-01") + pd.Timedelta(days=np.random.randint(0, 700)),
            None,
        ]) for _ in range(len(some_ids) + n)
    ],
    "age": np.where(np.random.rand(n + len(some_ids)) < .08, np.nan,
                    np.random.normal(40, 13, n + len(some_ids)).clip(18).round()),
    "monthly_charges": [messy_money(v) for v in
                        np.random.uniform(20, 120, n + len(some_ids))],
    "contract": np.random.choice(["Month-to-month", "month-to-month ",
                                  "MONTHLY", "One year", "Two year"],
                                 n + len(some_ids), p=[.5, .1, .1, .2, .1]),
    "payment_method": np.random.choice(
        ["credit card", "bank transfer", "e-wallet", "paypal", "crypto"],
        n + len(some_ids), p=[.4, .3, .2, .09, .01]),   # crypto = rare category
    "support_tickets": np.where(np.random.rand(n + len(some_ids)) < .10, np.nan,
                                np.random.poisson(1.2, n + len(some_ids))),
    "satisfaction_score": np.clip(np.round(np.random.normal(7, 2.2, n + len(some_ids))),
                                  1, 10).astype(float),
    "churn": np.random.choice(["Yes", "No"], n + len(some_ids), p=[.26, .74]),
})

feedback_pool = ["Great service!", "refund NOW!!", "ok", "support was slow...",
                 "love it", "price too high", "app keeps crashing"]
raw["feedback_text"] = np.random.choice(feedback_pool, n + len(some_ids))

# inject pathologies:
raw.loc[raw.sample(frac=0.02, random_state=1).index, "age"] = 250       # impossible age
raw.loc[raw.sample(frac=0.01, random_state=2).index, "satisfaction_score"] = 99
raw["signup_date"] = raw["signup_date"].astype(object)

# mixed date FORMATS as strings:
def fmt_date(x):
    if pd.isna(x):            # catches None AND NaT from the mixed column
        return np.nan
    style = np.random.choice(["iso", "eu", "text"])
    if style == "iso":
        return x.strftime("%Y-%m-%d")
    if style == "eu":
        return x.strftime("%d/%m/%Y")
    return x.strftime("%b %d %Y")

raw["signup_date"] = raw["signup_date"].apply(fmt_date)

df_raw = raw.copy()
print(df_raw.shape)
df_raw.head(8)

(312, 10)


,customer_id,signup_date,age,monthly_charges,contract,payment_method,support_tickets,satisfaction_score,churn,feedback_text
0,cust-000,2022-10-10,40.0,60.53,Month-to-month,credit card,1.0,7.0,No,love it
1,cust-001,2022-08-14,NaN,80.19,Month-to-month,credit card,2.0,8.0,Yes,Great service!
2,cust-002,NaN,38.0,83.25,Month-to-month,credit card,1.0,9.0,Yes,ok
3,cust-003,26/04/2023,31.0,57.28,Month-to-month,e-wallet,1.0,9.0,No,price too high
4,cust-004,NaN,64.0,71.26,Two year,credit card,0.0,10.0,No,love it
5,cust-005,NaN,40.0,61.30,Month-to-month,e-wallet,1.0,8.0,No,app keeps crashing
6,cust-006,NaN,28.0,26.91,Month-to-month,bank transfer,2.0,10.0,Yes,refund NOW!!
7,cust-007,NaN,NaN,63.72,Month-to-month,e-wallet,2.0,99.0,No,support was slow...


## Step 1 - INSPECT before touching anything

In [3]:
print(df_raw.info())
print("\nduplicated ids:", df_raw["customer_id"].duplicated().sum())
print("unique contracts:", df_raw["contract"].unique())

def missing_summary(df):
    m = pd.DataFrame({
        "n_missing": df.isna().sum(),
        "pct": (df.isna().mean()*100).round(1),
    })
    return m[m["n_missing"] > 0]

missing_summary(df_raw)

<class 'pandas.DataFrame'>
RangeIndex: 312 entries, 0 to 311
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   customer_id         312 non-null    str    
 1   signup_date         162 non-null    str    
 2   age                 294 non-null    float64
 3   monthly_charges     312 non-null    str    
 4   contract            312 non-null    str    
 5   payment_method      312 non-null    str    
 6   support_tickets     284 non-null    float64
 7   satisfaction_score  312 non-null    float64
 8   churn               312 non-null    str    
 9   feedback_text       312 non-null    str    
dtypes: float64(3), str(7)
memory usage: 24.5 KB
None

duplicated ids: 12
unique contracts: <StringArray>
['Month-to-month', 'Two year', 'One year', 'MONTHLY', 'month-to-month ']
Length: 5, dtype: str


,n_missing,pct
signup_date,150,48.1
age,18,5.8
support_tickets,28,9.0


## Step 2 - Normalize strings (casing/whitespace chaos)

In [4]:
def normalize_category(s: pd.Series) -> pd.Series:
    """strip -> collapse spaces -> Title-case; canonicalizes category labels."""
    return s.astype(str).str.strip().str.replace(r"\s+", " ", regex=True).str.title()

df = df_raw.copy()
print("before:", df["contract"].unique())
df["contract"] = normalize_category(df["contract"])
df["payment_method"] = normalize_category(df["payment_method"])
print("after :", sorted(df["contract"].unique()))

before: <StringArray>
['Month-to-month', 'Two year', 'One year', 'MONTHLY', 'month-to-month ']
Length: 5, dtype: str
after : ['Month-To-Month', 'Monthly', 'One Year', 'Two Year']


## Step 3 - Fix dtypes (dates & money)

In [5]:
def clean_money(s: pd.Series) -> pd.Series:
    """'$1,299.00' / '70.35' -> float64."""
    return pd.to_numeric(s.astype(str).str.replace(r"[$,]", "", regex=True),
                         errors="coerce")

df["monthly_charges"] = clean_money(df["monthly_charges"])

# Dates arrive in THREE formats. Parse each explicitly, then combine fills.
iso  = pd.to_datetime(df["signup_date"], format="%Y-%m-%d", errors="coerce")
eu   = pd.to_datetime(df["signup_date"], format="%d/%m/%Y", errors="coerce")
txt  = pd.to_datetime(df["signup_date"], format="%b %d %Y", errors="coerce")

df["signup_date"] = iso.fillna(eu).fillna(txt)
print("unparseable dates remaining:", df["signup_date"].isna().sum())

unparseable dates remaining: 150


## Step 4 - Duplicates by business key

In [6]:
before = len(df)
df = df.drop_duplicates(subset="customer_id", keep="last")   # last export wins
print(f"removed {before - len(df)} duplicate customer rows")

removed 12 duplicate customer rows


## Step 5 - Impossible values → missing

In [7]:
bad_age = df["age"] > 120
print("impossible ages found:", bad_age.sum())
df.loc[bad_age, "age"] = np.nan            # convert lie -> unknown, handle next
bad_sat = df["satisfaction_score"] > 10
print("impossible satisfaction:", bad_sat.sum())
df.loc[bad_sat, "satisfaction_score"] = np.nan

impossible ages found: 6
impossible satisfaction: 3


## Step 6 - Missing strategy PER COLUMN (justified!)

In [8]:
# contract      : categorical -> mode fill (tiny share missing)
# age           : numeric skewed-ish -> median; better: GROUP median by contract
# tickets       : NaN likely means ZERO tickets -> document assumption, fill 0
# satisfaction  : numeric -> median
mode_contract = df["contract"].mode()[0]
df["contract"] = df["contract"].fillna(mode_contract)

group_median_age = df.groupby("contract")["age"].transform("median")
df["age"] = df["age"].fillna(group_median_age)

df["support_tickets"] = df["support_tickets"].fillna(0)     # ASSUMPTION: no contact

df["satisfaction_score"] = df["satisfaction_score"].fillna(
    df["satisfaction_score"].median())

# signup_date: remaining unparseable/missing stamps -> MEDIAN date
# (dates are ordinal, so the median is a sensible "typical signup")
df["signup_date"] = df["signup_date"].fillna(df["signup_date"].median())

assert df.isna().sum().sum() == 0, "still have holes!"
print("all missing handled ✓")

all missing handled ✓


## Step 7 - Outliers via IQR fences (cap, don't delete)

In [9]:
def iqr_cap(s: pd.Series, k: float = 1.5) -> pd.Series:
    """Winsorize beyond Tukey fences Q1-k·IQR / Q3+k·IQR."""
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    lo, hi = q1 - k*iqr, q3 + k*iqr
    return s.clip(lo, hi)

for col in ["monthly_charges", "satisfaction_score"]:
    before_vals = df[col].copy()
    df[col] = iqr_cap(df[col])
    changed = (before_vals != df[col]).sum()
    print(f"{col}: capped {changed} values")

monthly_charges: capped 0 values
satisfaction_score: capped 7 values


## Step 8 - Encode categoricals

In [10]:
contract_map = {"Month-To-Month": 0, "One Year": 1, "Two Year": 2}
df["contract_ord"] = df["contract"].map(contract_map)          # ordinal

df["churn_flag"] = df["churn"].map({"No": 0, "Yes": 1})        # binary target

pay_dummies = pd.get_dummies(df["payment_method"], prefix="pay",
                             drop_first=True)                  # nominal

# rare-category guard BEFORE dummies (crypto ~1%):
counts = df["payment_method"].value_counts()
rare = counts[counts < counts.sum()*0.02].index
df["payment_method_g"] = np.where(df["payment_method"].isin(rare), "Other",
                                  df["payment_method"])
pay_dummies = pd.get_dummies(df["payment_method_g"], prefix="pay", drop_first=True)

df_encoded = pd.concat([df.drop(columns=["contract", "churn",
                                         "payment_method", "payment_method_g"]),
                        pay_dummies], axis=1)
df_encoded.dtypes.value_counts()

float64           5
bool              4
str               2
datetime64[us]    1
int64             1
Name: count, dtype: int64

## Step 9 - Text features (light touch)

In [11]:
df_encoded["fb_len"] = df_encoded["feedback_text"].str.len()
df_encoded["fb_words"] = df_encoded["feedback_text"].str.split().str.len()
df_encoded["fb_exclaims"] = df_encoded["feedback_text"].str.count("!")
df_encoded["fb_urgent"] = df_encoded["feedback_text"].str.contains(
    "refund|now|crash", case=False).astype(int)

df_encoded.drop(columns="feedback_text").filter(like="fb_").describe().loc[["mean","max"]]

,fb_len,fb_words,fb_exclaims,fb_urgent
mean,12.206667,2.27,0.406667,0.306667
max,19.000000,3.00,2.000000,1.000000


## Step 10 - Scaling, WITHOUT leakage

In [12]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

feature_cols = ["age", "monthly_charges", "support_tickets",
                "satisfaction_score", "contract_ord", "fb_len",
                "fb_words", "fb_exclaims", "fb_urgent"]
X = df_encoded[feature_cols]
y = df_encoded["churn_flag"]

# ⚠️ WRONG WAY (leakage): scaler.fit(X) on ALL data lets test statistics into train.
# ✅ RIGHT WAY: fit on TRAIN only, transform both sides.
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42)

scaler = StandardScaler()
X_tr_scaled = pd.DataFrame(scaler.fit_transform(X_tr), columns=feature_cols)
X_te_scaled = pd.DataFrame(scaler.transform(X_te), columns=feature_cols)

print("train mean≈0/std≈1:", np.allclose(X_tr_scaled.mean(), 0, atol=1e-8))
print("test stats differ (as they should):")
print(X_te_scaled.describe().loc[["mean", "std"]].abs().round(2).iloc[:, :4])

train mean≈0/std≈1: True
test stats differ (as they should):
       age  monthly_charges  support_tickets  satisfaction_score
mean  0.11             0.18             0.12                0.17
std   0.99             1.08             0.90                1.10


## Save the artifact + reusable utilities recap

In [13]:
df_encoded.to_csv("processed_churn.csv", index=False)
print("saved processed_churn.csv:", df_encoded.shape)

# Utilities defined in this notebook - copy them to your own toolkit:
#   normalize_category(s)  canonicalize label columns
#   clean_money(s)         '$1,23.45' strings -> floats
#   iqr_cap(s, k=1.5)      winsorize outliers at Tukey fences
#   missing_summary(df)    count+percentage report of holes

saved processed_churn.csv: (300, 17)


## THE MASTER CHECKLIST (print me!)

| # | Step | Key tool | Watch out for |
#|---|---|---|---|
#|0| work on a COPY | `df = raw.copy()` | mutating source data |
#|1| inspect | `info/describe/nunique` | assuming dtypes |
#|2| normalize strings | `.str.strip().title()` | invisible trailing spaces |
#|3| fix dtypes | `to_datetime/to_numeric coerce` | silent NaTs - COUNT them |
#|4| dedupe by key | `drop_duplicates(subset)` | which record wins |
#|5| impossible values | domain rules -> NaN | deleting instead of flagging |
#|6| missing per column | mode/median/group-median/ffill | one-size-fits-all filling |
#|7| outliers | IQR caps / transforms | deleting real signal |
#|8| encode | map dicts, get_dummies(+rare guard) | fake ordinality, unseen cats |
#|9| scale LAST, fit on train | `train_test_split` first | LEAKAGE |

> 💡 **Pro tip:** order matters because later steps assume earlier ones: you can't
impute sensibly before normalizing categories, can't cap outliers before fixing
types, must never scale before splitting.

## Summary & key takeaways

- A fixed checklist turns chaotic cleaning into a reproducible procedure.
- Count every coercion (`errors="coerce"`) - failures are data-quality findings.
- Justify each imputation per column; group-wise medians beat global ones when
  structure exists.
- Cap outliers rather than delete unless provably erroneous input.
- Rare categories get grouped before encoding to keep dummy tables stable.
- **Split FIRST, fit scalers/encoders on train only** - the cardinal rule.
- Save the processed frame; your future self (and ML track) will reuse it.